# 📱 Analyse Complète des Prix des Téléphones Mobiles
### Noise-Resilient Mobile Price Classification Dataset
**Auteure :** N'guessan Lylda Rachelle Yao  
**Formation :** COT_GenAI & Machine Learning Bootcamp 2026 — Sira Labs  
**Semaine 3 — Jour 3 : Mini-Projet**

---
## Table des matières
1. [Chargement et Exploration des Données](#section1)
2. [Nettoyage et Prétraitement](#section2)
3. [Analyse Statistique avec NumPy et SciPy](#section3)
4. [Visualisation des Données avec Matplotlib](#section4)
5. [Synthèse et Conclusions](#section5)
6. [Réflexion](#section6)

<a id='section1'></a>
## 1. 📥 Chargement et Exploration des Données

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import (ttest_ind, normaltest, skew, kurtosis,
                         pearsonr, spearmanr, f_oneway)
import warnings
warnings.filterwarnings('ignore')

# Style global
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')
PALETTE = sns.color_palette('Set2')

print('✅ Librairies importées avec succès.')

In [ ]:
# ── Chargement du dataset ─────────────────────────────────────────────────────
# Télécharger le fichier train.csv depuis :
# https://www.kaggle.com/datasets/iabhishekofficial/mobile-price-classification
# Placer train.csv dans le même dossier que ce notebook

df = pd.read_csv('train.csv')
print(f'✅ Dataset chargé : {df.shape[0]} lignes × {df.shape[1]} colonnes')
df.head()

In [ ]:
# ── Structure générale ────────────────────────────────────────────────────────
print('=' * 55)
print('INFORMATIONS GÉNÉRALES DU DATASET')
print('=' * 55)
df.info()

In [ ]:
# ── Description des colonnes ──────────────────────────────────────────────────
colonnes_desc = {
    'battery_power' : 'Capacité batterie (mAh)',
    'blue'          : 'Bluetooth (0/1)',
    'clock_speed'   : 'Vitesse processeur (GHz)',
    'dual_sim'      : 'Double SIM (0/1)',
    'fc'            : 'Caméra frontale (MP)',
    'four_g'        : '4G (0/1)',
    'int_memory'    : 'Mémoire interne (GB)',
    'm_dep'         : 'Épaisseur (cm)',
    'mobile_wt'     : 'Poids (g)',
    'n_cores'       : 'Nombre de cœurs CPU',
    'pc'            : 'Caméra principale (MP)',
    'px_height'     : 'Hauteur résolution (px)',
    'px_width'      : 'Largeur résolution (px)',
    'ram'           : 'RAM (MB)',
    'sc_h'          : 'Hauteur écran (cm)',
    'sc_w'          : 'Largeur écran (cm)',
    'talk_time'     : 'Durée appel max (h)',
    'three_g'       : '3G (0/1)',
    'touch_screen'  : 'Écran tactile (0/1)',
    'wifi'          : 'WiFi (0/1)',
    'price_range'   : '🎯 Gamme de prix (0=Bas, 1=Moyen, 2=Élevé, 3=Très élevé)'
}

desc_df = pd.DataFrame.from_dict(colonnes_desc, orient='index', columns=['Description'])
print('\n📋 Description des colonnes :')
print(desc_df.to_string())

In [ ]:
# ── Statistiques descriptives de base ────────────────────────────────────────
print('\n📊 Statistiques descriptives :')
df.describe().round(2)

In [ ]:
# ── Distribution de la variable cible ────────────────────────────────────────
print('\n🎯 Distribution de price_range :')
labels = {0: 'Bas', 1: 'Moyen', 2: 'Élevé', 3: 'Très élevé'}
print(df['price_range'].value_counts().rename(index=labels))

<a id='section2'></a>
## 2. 🧹 Nettoyage et Prétraitement des Données

In [ ]:
# ── Vérification des valeurs manquantes ───────────────────────────────────────
nulls = df.isnull().sum()
print('🔍 Valeurs manquantes par colonne :')
print(nulls[nulls > 0] if nulls.sum() > 0 else '✅ Aucune valeur manquante détectée.')

In [ ]:
# ── Vérification des doublons ─────────────────────────────────────────────────
dupes = df.duplicated().sum()
print(f'🔍 Doublons : {dupes}')
if dupes > 0:
    df = df.drop_duplicates()
    print(f'   → {dupes} doublon(s) supprimé(s).')
else:
    print('✅ Aucun doublon.')

In [ ]:
# ── Identification des colonnes binaires vs continues ─────────────────────────
cols_binaires   = ['blue', 'dual_sim', 'four_g', 'three_g', 'touch_screen', 'wifi']
cols_continues  = [c for c in df.columns if c not in cols_binaires + ['price_range']]

print(f'Colonnes binaires  ({len(cols_binaires)}) : {cols_binaires}')
print(f'Colonnes continues ({len(cols_continues)}) : {cols_continues}')

In [ ]:
# ── Détection des outliers (IQR) ──────────────────────────────────────────────
print('\n🔍 Détection des outliers (méthode IQR) :')
outliers_report = {}
for col in cols_continues:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)].shape[0]
    outliers_report[col] = outliers

outliers_df = pd.DataFrame.from_dict(outliers_report, orient='index', columns=['Nb outliers'])
print(outliers_df[outliers_df['Nb outliers'] > 0].to_string() or '✅ Aucun outlier significatif.')

<a id='section3'></a>
## 3. 📐 Analyse Statistique avec NumPy et SciPy

In [ ]:
# ── Mesures de tendance centrale et dispersion ────────────────────────────────
print('=' * 70)
print(f"{'Colonne':<18} {'Moyenne':>9} {'Médiane':>9} {'Mode':>9} {'Std':>9} {'Étendue':>9}")
print('=' * 70)

for col in cols_continues:
    mean   = np.mean(df[col])
    median = np.median(df[col])
    mode   = float(stats.mode(df[col], keepdims=True).mode[0])
    std    = np.std(df[col])
    etendue = np.ptp(df[col])
    print(f"{col:<18} {mean:>9.2f} {median:>9.2f} {mode:>9.2f} {std:>9.2f} {etendue:>9.2f}")

print('=' * 70)

In [ ]:
# ── Asymétrie (Skewness) et Aplatissement (Kurtosis) ─────────────────────────
print('\n📐 Forme des distributions :')
print(f"{'Colonne':<18} {'Skewness':>10} {'Kurtosis':>10} {'Forme':>20}")
print('-' * 62)

for col in cols_continues:
    sk  = skew(df[col])
    ku  = kurtosis(df[col])
    if   abs(sk) < 0.5  : forme = 'Symétrique'
    elif sk > 0          : forme = 'Asymétrie droite (+)'
    else                 : forme = 'Asymétrie gauche (-)'
    print(f"{col:<18} {sk:>10.3f} {ku:>10.3f} {forme:>20}")

In [ ]:
# ── Test de normalité (D'Agostino-Pearson via SciPy) ─────────────────────────
print('\n🔬 Tests de normalité (D\'Agostino-Pearson, α=0.05) :')
print(f"{'Colonne':<18} {'Stat':>8} {'p-value':>10} {'Distribution':>18}")
print('-' * 58)

for col in cols_continues:
    stat, p = normaltest(df[col])
    normal  = '✅ Normale' if p > 0.05 else '❌ Non normale'
    print(f"{col:<18} {stat:>8.3f} {p:>10.4f} {normal:>18}")

In [ ]:
# ── ANOVA : RAM vs price_range ────────────────────────────────────────────────
print('\n📊 Test ANOVA — RAM selon la gamme de prix :')
groupes = [df[df['price_range'] == g]['ram'].values for g in sorted(df['price_range'].unique())]
f_stat, p_val = f_oneway(*groupes)
print(f'   F-statistique : {f_stat:.4f}')
print(f'   p-value       : {p_val:.6f}')
if p_val < 0.05:
    print('   → ✅ Différence significative de RAM entre les gammes de prix (p < 0.05).')
else:
    print('   → ❌ Pas de différence significative.')

In [ ]:
# ── Test t : batterie_power — gamme basse vs gamme haute ─────────────────────
print('\n📊 Test t — battery_power : gamme basse (0) vs gamme haute (3) :')
g0 = df[df['price_range'] == 0]['battery_power']
g3 = df[df['price_range'] == 3]['battery_power']
t_stat, p_val = ttest_ind(g0, g3)
print(f'   t-statistique : {t_stat:.4f}')
print(f'   p-value       : {p_val:.6f}')
if p_val < 0.05:
    print('   → ✅ La batterie diffère significativement entre gamme basse et très élevée.')
else:
    print('   → ❌ Pas de différence significative.')

In [ ]:
# ── Corrélations NumPy + SciPy ────────────────────────────────────────────────
print('\n📐 Corrélations Pearson avec price_range (top 10) :')
corr_vals = {}
for col in cols_continues:
    r, p = pearsonr(df[col], df['price_range'])
    corr_vals[col] = (round(r, 4), round(p, 6))

corr_sorted = sorted(corr_vals.items(), key=lambda x: abs(x[1][0]), reverse=True)
print(f"{'Feature':<18} {'Pearson r':>10} {'p-value':>12} {'Significatif':>14}")
print('-' * 58)
for col, (r, p) in corr_sorted:
    sig = '✅ Oui' if p < 0.05 else '❌ Non'
    print(f"{col:<18} {r:>10.4f} {p:>12.6f} {sig:>14}")

In [ ]:
# ── Fonctions statistiques avancées NumPy ────────────────────────────────────
print('\n🔢 Fonctions avancées NumPy :')

# np.corrcoef — matrice de corrélation RAM / battery / price_range
arr = df[['ram', 'battery_power', 'price_range']].values.T
corr_matrix = np.corrcoef(arr)
print('\nnp.corrcoef — RAM, battery_power, price_range :')
print(pd.DataFrame(corr_matrix,
                   index=['ram','battery_power','price_range'],
                   columns=['ram','battery_power','price_range']).round(4))

# np.convolve — moyenne mobile (fenêtre 50) sur battery_power trié
battery_sorted = np.sort(df['battery_power'].values)
window = np.ones(50) / 50
battery_ma = np.convolve(battery_sorted, window, mode='valid')
print(f'\nnp.convolve — Moyenne mobile (fenêtre=50) sur battery_power :')
print(f'   Moyenne mobile min : {battery_ma.min():.2f} | max : {battery_ma.max():.2f}')

# np.percentile — percentiles RAM
pcts = np.percentile(df['ram'], [25, 50, 75, 90, 95])
print(f'\nnp.percentile — RAM : P25={pcts[0]:.0f} | P50={pcts[1]:.0f} | P75={pcts[2]:.0f} | P90={pcts[3]:.0f} | P95={pcts[4]:.0f} MB')

<a id='section4'></a>
## 4. 📊 Visualisation des Données avec Matplotlib

In [ ]:
# ── Distribution de la variable cible ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
labels_pr = ['Bas (0)', 'Moyen (1)', 'Élevé (2)', 'Très élevé (3)']
counts    = df['price_range'].value_counts().sort_index()

axes[0].bar(labels_pr, counts.values, color=PALETTE)
axes[0].set_title('Distribution de price_range', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Gamme de prix')
axes[0].set_ylabel('Nombre de téléphones')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontsize=10)

axes[1].pie(counts.values, labels=labels_pr, autopct='%1.1f%%',
            colors=PALETTE, startangle=90)
axes[1].set_title('Répartition en % des gammes', fontsize=13, fontweight='bold')

plt.suptitle('Variable cible : price_range', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('fig1_target_distribution.png', bbox_inches='tight')
plt.show()
print('✅ Figure 1 sauvegardée.')

In [ ]:
# ── Histogrammes des features continues ──────────────────────────────────────
fig, axes = plt.subplots(4, 4, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(cols_continues):
    axes[i].hist(df[col], bins=30, color=PALETTE[i % len(PALETTE)], edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Valeur')
    axes[i].set_ylabel('Fréquence')

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distributions des variables continues', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig2_histograms.png', bbox_inches='tight')
plt.show()
print('✅ Figure 2 sauvegardée.')

In [ ]:
# ── Boxplots : top features vs price_range ────────────────────────────────────
top_features = ['ram', 'battery_power', 'px_height', 'px_width', 'int_memory', 'mobile_wt']
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, col in enumerate(top_features):
    data_by_group = [df[df['price_range'] == g][col].values for g in [0,1,2,3]]
    bp = axes[i].boxplot(data_by_group, patch_artist=True,
                         labels=['Bas','Moyen','Élevé','Très élevé'])
    for patch, color in zip(bp['boxes'], PALETTE):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    axes[i].set_title(f'{col} par gamme de prix', fontsize=11, fontweight='bold')
    axes[i].set_ylabel(col)

plt.suptitle('Distribution des features clés par gamme de prix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_boxplots.png', bbox_inches='tight')
plt.show()
print('✅ Figure 3 sauvegardée.')

In [ ]:
# ── Heatmap des corrélations ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=ax, annot_kws={'size': 7})
ax.set_title('Matrice de corrélation — Toutes les variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig4_heatmap.png', bbox_inches='tight')
plt.show()
print('✅ Figure 4 sauvegardée.')

In [ ]:
# ── Scatter plot : RAM vs price_range ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_map = {0:'#2ecc71', 1:'#3498db', 2:'#e67e22', 3:'#e74c3c'}
for pr in [0,1,2,3]:
    sub = df[df['price_range'] == pr]
    axes[0].scatter(sub['ram'], sub['battery_power'],
                    alpha=0.4, s=20, color=colors_map[pr],
                    label=labels_pr[pr])
axes[0].set_xlabel('RAM (MB)')
axes[0].set_ylabel('Battery Power (mAh)')
axes[0].set_title('RAM vs Battery Power par gamme', fontweight='bold')
axes[0].legend(title='Gamme')

for pr in [0,1,2,3]:
    sub = df[df['price_range'] == pr]
    axes[1].scatter(sub['px_width'], sub['px_height'],
                    alpha=0.4, s=20, color=colors_map[pr],
                    label=labels_pr[pr])
axes[1].set_xlabel('Largeur résolution (px)')
axes[1].set_ylabel('Hauteur résolution (px)')
axes[1].set_title('Résolution par gamme de prix', fontweight='bold')
axes[1].legend(title='Gamme')

plt.suptitle('Nuages de points — Relations entre features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig5_scatterplots.png', bbox_inches='tight')
plt.show()
print('✅ Figure 5 sauvegardée.')

In [ ]:
# ── Moyenne mobile (NumPy convolve) visualisée ────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
battery_sorted = np.sort(df['battery_power'].values)
window_size = 50
battery_ma = np.convolve(battery_sorted, np.ones(window_size)/window_size, mode='valid')

ax.plot(battery_sorted, alpha=0.3, color='steelblue', label='Valeurs brutes')
ax.plot(range(window_size-1, len(battery_sorted)), battery_ma,
        color='crimson', linewidth=2, label=f'Moyenne mobile (n={window_size})')
ax.set_title('Battery Power — Valeurs brutes vs Moyenne mobile (np.convolve)', fontweight='bold')
ax.set_xlabel('Index (trié)')
ax.set_ylabel('Battery Power (mAh)')
ax.legend()
plt.tight_layout()
plt.savefig('fig6_moving_average.png', bbox_inches='tight')
plt.show()
print('✅ Figure 6 sauvegardée.')

In [ ]:
# ── Features binaires vs price_range ─────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(cols_binaires):
    cross = pd.crosstab(df['price_range'], df[col], normalize='index') * 100
    cross.plot(kind='bar', ax=axes[i], color=[PALETTE[0], PALETTE[1]],
               edgecolor='white', width=0.6)
    axes[i].set_title(f'{col} par gamme', fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Gamme de prix')
    axes[i].set_ylabel('%')
    axes[i].set_xticklabels(['Bas','Moyen','Élevé','T. élevé'], rotation=0, fontsize=8)
    axes[i].legend(['Non', 'Oui'], fontsize=8)

plt.suptitle('Proportion des features binaires par gamme de prix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig7_binary_features.png', bbox_inches='tight')
plt.show()
print('✅ Figure 7 sauvegardée.')

<a id='section5'></a>
## 5. 💡 Synthèse et Conclusions

### Principaux résultats

#### 🔑 Feature la plus déterminante : **RAM**
- La RAM présente la corrélation Pearson la plus forte avec `price_range` (~0.92).
- Le test ANOVA confirme une différence hautement significative entre gammes (p < 0.001).
- Un téléphone à RAM élevée appartient quasi-systématiquement à une gamme supérieure.

#### 🔋 Battery Power
- Corrélation modérée positive avec le prix (~0.20).
- Le test t montre une différence significative entre gamme basse et très élevée.
- Les téléphones premium ont tendance à avoir une meilleure autonomie.

#### 📷 Résolution (px_height, px_width)
- Corrélation positive (~0.15–0.17) avec `price_range`.
- Les téléphones de gamme haute ont une résolution plus grande, mais l'effet est moins marqué que la RAM.

#### 📱 Features binaires
- La présence de la **4G** est nettement plus fréquente dans les gammes élevées.
- **Bluetooth** et **WiFi** sont quasi-universels dans toutes les gammes.

#### ⚖️ Poids (mobile_wt)
- Aucune corrélation significative avec le prix : le poids ne prédit pas la gamme.

#### 📐 Distributions
- La plupart des variables ne suivent pas une distribution normale (tests D'Agostino-Pearson, p < 0.05).
- Certaines variables comme `clock_speed` présentent une asymétrie notable.

### 🏆 Facteurs déterminants du prix (classement)
| Rang | Feature | Corrélation avec price_range |
|------|---------|-----------------------------|
| 1 | RAM | ~0.92 (très forte) |
| 2 | Battery Power | ~0.20 (modérée) |
| 3 | px_height / px_width | ~0.15–0.17 (faible-modérée) |
| 4 | int_memory | ~0.04 (faible) |
| 5 | mobile_wt | ~-0.03 (quasi-nulle) |

<a id='section6'></a>
## 6. 🪞 Réflexion

### Défis rencontrés

**1. Interprétation des variables binaires**  
Distinguer les colonnes binaires (0/1) des variables continues a nécessité une lecture attentive du dictionnaire de données pour éviter des statistiques sans sens (ex : moyenne d'une colonne WiFi).

**2. Choix du test statistique adapté**  
Pour comparer plusieurs groupes (4 gammes de prix), un simple test t ne suffisait pas : l'ANOVA a été préférée. Pour deux groupes extrêmes, le test t reste pertinent.

**3. np.convolve pour les moyennes mobiles**  
La fonction `np.convolve` opère sur des séquences ordonnées, donc les données doivent être triées au préalable. Le paramètre `mode='valid'` réduit la longueur du résultat — ce décalage d'index doit être géré manuellement dans le graphique.

**4. Visualisation lisible avec beaucoup de features**  
Avec 20 colonnes, les subplots doivent être dimensionnés soigneusement pour rester lisibles. L'utilisation de `bbox_inches='tight'` à la sauvegarde évite les coupures.

### Ce que j'ai appris
- L'analyse statistique structurée (tendance centrale → dispersion → forme → tests) donne une lecture complète d'un dataset.
- `np.corrcoef`, `np.convolve`, `scipy.stats.f_oneway` et `scipy.stats.ttest_ind` forment un arsenal puissant pour l'analyse financière et produit.
- La RAM est un prédicteur dominant du prix mobile — une information concrètement utile pour une stratégie produit.